# MoE Backbone Tests on BDD100K

Testing the new MoE architectures (WP-M0) on actual BDD100K data streams. We will visualize how router probabilities differ between different driving conditions.
Run from the repository root or `notebooks/`. Start Jupyter with the project dependencies:

```powershell
uv run --extra cpu --with pandas --with matplotlib --with seaborn --with tqdm --with jupyterlab jupyter lab
```

Use `--extra cu124` instead of `--extra cpu` for the repository's CUDA environment. Set `BDD_ROOT` (or edit `bdd_root`) to the BDD100K directory. Set `MOE_CHECKPOINT` to a compatible encoder/full-method checkpoint for pretrained experiments; without it the notebook explicitly uses random initialization and trains the full backbone. The default 2,000-image cap bounds RAM use. `training_epochs`, `training_batches`, and `activation_iterations` control runtime.


In [ ]:
import os
import sys
from pathlib import Path

repo_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "cafl4ds/configs").is_dir()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from within the cafl4ds repository.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from hydra.utils import instantiate
from loguru import logger
from omegaconf import OmegaConf

from cafl4ds.models.vit import MoEViTEncoder

# BDD100K configuration
IMAGE_SIZE = 64  # Change this variable to control image resolution
PATCH_SIZE = 8  # Match pretrained weights

bdd_root = Path(os.environ.get("BDD_ROOT", r"D:\AI Proj\bdd100k_images_100k")).expanduser()
bdd_split = "train"
images_dir = bdd_root / "images" / "100k" / bdd_split
labels_file = bdd_root / "labels" / f"bdd100k_labels_images_{bdd_split}.json"

source_cfg = OmegaConf.create(
    {
        "_target_": "cafl4ds.data.attributes.BDD100KSource",
        "bdd_root": str(bdd_root),
        "split": bdd_split,
        "images_dir": str(images_dir),
        "labels_file": str(labels_file),
        "img_size": IMAGE_SIZE,  # small size for quick testing
        "max_images": 2000,  # set to None to use all images
        "min_canary_count": 32,
    }
)

stream_cfg = OmegaConf.create(
    {
        "_target_": "cafl4ds.data.regime.RegimeStream",
        "batch_size": 128,
        "support_per_canary": 8,
        "query_per_canary": 8,
        "max_train_per_regime": 128,
        "canary_seed": 0,
    }
)
# The source caches decoded images in RAM; increase the cap deliberately.
source_cfg.legacy_labels = not labels_file.is_dir()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Set MOE_CHECKPOINT or this variable to an optional compatible encoder/full-method state dict.
# None explicitly runs a random-start experiment; pretrained claims do not apply.
checkpoint_path = Path(os.environ["MOE_CHECKPOINT"]).expanduser() if os.environ.get("MOE_CHECKPOINT") else None
training_epochs = 20
training_batches = 150
activation_iterations = 300


def initialize_encoder(encoder):
    if checkpoint_path is None:
        logger.warning("Random initialization: no MOE_CHECKPOINT supplied; the whole backbone is trainable.")
        return
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    state = {k.removeprefix("encoder."): v for k, v in checkpoint.items() if k.startswith("encoder.")}
    if not state:
        state = dict(checkpoint)  # Also accept an encoder-only state dict.
    if "pos_embed" in state and state["pos_embed"].shape != encoder.pos_embed.shape:
        import torch.nn.functional as F

        positions = state["pos_embed"]
        grid = int((positions.shape[1] - 1) ** 0.5)
        spatial = positions[:, 1:].reshape(1, grid, grid, -1).permute(0, 3, 1, 2)
        spatial = F.interpolate(
            spatial, size=(encoder.grid_size, encoder.grid_size), mode="bicubic", align_corners=False
        )
        state["pos_embed"] = torch.cat([positions[:, :1], spatial.flatten(2).transpose(1, 2)], dim=1)
    missing, unexpected = encoder.load_state_dict(state, strict=False)
    # A dense checkpoint may omit MoE MLPs, but must cover every shared backbone parameter.
    moe_prefixes = tuple(f"blocks.{i}.mlp." for i in encoder.moe_blocks)
    missing_shared = [key for key in missing if not key.startswith(moe_prefixes)]
    if missing_shared:
        raise ValueError(f"Checkpoint is missing shared backbone weights: {missing_shared}")
    logger.info("Loaded checkpoint {}; missing: {}; unexpected: {}", checkpoint_path, missing, unexpected)
    for name, param in encoder.named_parameters():
        param.requires_grad_(name.startswith(tuple(f"blocks.{i}." for i in encoder.moe_blocks)))

In [ ]:
source = instantiate(source_cfg)
stream = instantiate(stream_cfg, source=source, seed=13)
if not len(stream):
    raise ValueError("No training batches remain; increase max_images or reduce probe reservations.")
logger.info("Loaded {} batches; conditions: {}", len(stream), stream.era_names)

## Visualizing MoE Routing on BDD100K

Here we test the MoE Backbone routing logic on the BDD100K stream. We initialize our `MoETinyViTEncoder` and feed it a batch of images to check which experts process which tokens.

In [ ]:
torch.manual_seed(13)  # Paired initialization for the balancing ablation.
number_of_experts = 4

# --- Change this variable to test different Regimes ---
# e.g. 'daytime', 'night', 'dawn/dusk'
target_regime_names = ["daytime", "night", "dawn/dusk"]


# Initialize the MoE Encoder
encoder = MoEViTEncoder(
    img_size=IMAGE_SIZE,
    patch_size=PATCH_SIZE,
    in_chans=3,
    embed_dim=64,
    depth=4,
    num_heads=2,
    mlp_ratio=2.0,
    num_experts=number_of_experts,
    top_k=1,
    moe_blocks=(2, 3),
    routing_level="token",
    balancing_mode="batch",
)

initialize_encoder(encoder)
encoder = encoder.to(device)
encoder.eval()


if stream is not None and len(stream) > 0:
    target_era = 0
    if hasattr(stream, "regime_names") and hasattr(stream, "_regime_order"):
        try:
            # stream.regime_names is a dict[int, str]
            reg_id = next(k for k, v in stream.regime_names.items() if target_regime_names[0] in v)
            target_era = stream._regime_order.index(reg_id)
        except (ValueError, StopIteration):
            logger.warning(f"Regime '{target_regime_names}' not found. Using first batch.")

    batch = next((b for b in stream if b.era == target_era), None)
    if batch is None:
        batch = next(iter(stream))
    device = next(encoder.parameters()).device
    imgs = batch.images.to(device)
    logger.info(f"Processing batch for '{target_regime_names}' with {imgs.shape[0]} images.")
else:
    # Fallback to dummy data if dataset is not found
    raise ValueError("Stream failed to load! Please check the BDD100K data path and memory usage.")

# Forward pass to get routing probabilities
with torch.no_grad():
    embeds, aux_loss = encoder.embed_with_aux(imgs)
probs = encoder.routing(imgs)
print(f"Auxiliary balancing loss: {aux_loss:.4f}")

### Routing Probabilities Visualization

We visualize the routing probabilities for each MoE block. A heatmap illustrates how tokens (y-axis) are distributed among the 4 experts (x-axis).

In [ ]:
probs_block2 = probs[encoder.moe_blocks[0]].detach().cpu().numpy()  # [B, T, E]
probs_block3 = probs[encoder.moe_blocks[1]].detach().cpu().numpy()  # [B, T, E]

sample_idx = 0  # Visualize the first image in the batch

# Calculate number of tokens and adjust annotations dynamically
num_tokens = probs_block2.shape[1]
show_annot = num_tokens <= 32
yticklabels = [f"{i}" for i in range(num_tokens)] if show_annot else False
fig_height = max(6, int(num_tokens * 0.3)) if show_annot else 8

fig, axes = plt.subplots(1, 2, figsize=(14, fig_height))

sns.heatmap(
    probs_block2[sample_idx],
    cmap="Blues",
    ax=axes[0],
    cbar_kws={"label": "Probability"},
    annot=show_annot,
    fmt=".2f",
    yticklabels=yticklabels,
)
axes[0].set_title("Encoder Layer 3 Routing (First Image)")
axes[0].set_xlabel("Expert Index")
axes[0].set_ylabel("Token Index (Spatial Patch)")

sns.heatmap(
    probs_block3[sample_idx],
    cmap="Oranges",
    ax=axes[1],
    cbar_kws={"label": "Probability"},
    annot=show_annot,
    fmt=".2f",
    yticklabels=yticklabels,
)
axes[1].set_title("Encoder Layer 4 Routing (First Image)")
axes[1].set_xlabel("Expert Index")
axes[1].set_ylabel("Token Index (Spatial Patch)")

plt.tight_layout()
plt.show()

### Spatial Routing Distribution (GradCAM-like)

We project the expert routing probabilities back to the spatial dimensions of the image. This allows us to see which parts of the image each expert specializes in processing.

In [ ]:
import math

import numpy as np
import torch.nn.functional as F

sample_idx = 0
img = imgs[sample_idx].cpu().detach().cpu().numpy().transpose(1, 2, 0)
# Normalize image for visualization
img = (img - img.min()) / (img.max() - img.min() + 1e-8)

# We visualize the spatial routing for the first MoE block
p_block = probs[encoder.moe_blocks[0]][sample_idx].detach().cpu()  # [T, E]
num_patches = (IMAGE_SIZE // PATCH_SIZE) ** 2
if p_block.shape[0] > num_patches:
    p_block = p_block[-num_patches:]  # Keep only spatial patches if class token is present

E = p_block.shape[1]
grid_size = int(np.sqrt(p_block.shape[0]))
img_size = imgs.shape[2:]

cols = math.ceil(E / 2)
fig, axes = plt.subplots(2, cols, figsize=(3 * cols, 6))
axes = axes.flatten()

vmax_val = p_block.max().item()  # Dynamic vmax for stronger heatmap

for e in range(E):
    expert_prob = p_block[:, e].view(1, 1, grid_size, grid_size)
    expert_prob_resized = F.interpolate(expert_prob, size=img_size, mode="bilinear", align_corners=False)
    expert_prob_resized = expert_prob_resized.squeeze().numpy()

    ax = axes[e]
    ax.imshow(img)
    im = ax.imshow(expert_prob_resized, cmap="jet", alpha=0.65, vmin=0, vmax=vmax_val)
    ax.set_title(f"Expert {e}")
    ax.axis("off")

# Hide any unused subplots
for e in range(E, len(axes)):
    axes[e].axis("off")

cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cbar_ax, label="Routing Probability")
plt.suptitle("Spatial Routing Distribution for First MoE Block")
plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.show()

### Mean Expert Usage

Let's look at the overall expert usage across the entire batch.

In [ ]:
# Compute average expert load across the batch and all tokens
mean_usage_block2 = probs_block2.mean(axis=(0, 1))
mean_usage_block3 = probs_block3.mean(axis=(0, 1))

fig, ax = plt.subplots(figsize=(8, 5))
width = 0.35
x = range(number_of_experts)

ax.bar([i - width / 2 for i in x], mean_usage_block2, width, label="Block 2", color="steelblue")
ax.bar([i + width / 2 for i in x], mean_usage_block3, width, label="Block 3", color="darkorange")

ax.set_xlabel("Expert Index")
ax.set_ylabel("Average Routing Probability")
ax.set_title("Mean Expert Usage across Batch")
ax.set_xticks(x)
ax.set_xticklabels([f"{i}" for i in x])
ax.legend()
plt.show()

## Regime Routing Comparison

Here we systematically compare how different regimes (e.g. daytime vs night vs dawn/dusk) influence the expert routing distribution.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

regimes_to_test = ["daytime", "night", "dawn/dusk"]
expert_counts_per_regime = {}

if stream is not None:
    for regime_name in regimes_to_test:
        try:
            reg_id = next(k for k, v in stream.regime_names.items() if regime_name in v)
            target_era = stream._regime_order.index(reg_id)
            batch = next((b for b in stream if b.era == target_era), None)
            if batch is not None:
                device = next(encoder.parameters()).device
                imgs = batch.images.to(device)
                with torch.no_grad():
                    probs = encoder.routing(imgs)
                # probs is a list of tensors of shape [batch_size * seq_len, num_experts]
                agg_probs = []
                for prob in probs.values():
                    avg_prob = prob.mean(dim=(0, 1)).cpu().numpy()
                    agg_probs.append(avg_prob)
                expert_counts_per_regime[regime_name] = agg_probs
        except (ValueError, StopIteration):
            logger.warning(f"Regime containing '{regime_name}' not found in stream.")

if expert_counts_per_regime:
    num_blocks = len(next(iter(expert_counts_per_regime.values())))
    fig, axes = plt.subplots(1, num_blocks, figsize=(6 * num_blocks, 5))
    if num_blocks == 1:
        axes = [axes]

    for i in range(num_blocks):
        ax = axes[i]
        num_experts = len(next(iter(expert_counts_per_regime.values()))[i])
        x = np.arange(num_experts)
        width = 0.25
        multiplier = 0

        for regime_name, agg_probs in expert_counts_per_regime.items():
            offset = width * multiplier
            rects = ax.bar(x + offset, agg_probs[i], width, label=regime_name)
            multiplier += 1

        ax.set_title(f"Average Expert Probabilities (MoE Block {i})")
        ax.set_xlabel("Expert Index")
        ax.set_ylabel("Average Probability")
        ax.set_xticks(x + width * (len(expert_counts_per_regime) - 1) / 2)
        ax.set_xticklabels([f"E{j}" for j in range(num_experts)])
        ax.legend(loc="upper right")

    plt.tight_layout()
    plt.show()
else:
    print("No valid regimes found for comparison.")

## Training the MoE Encoder via SimSiam

We hook up our `MoETinyViTEncoder` into the `SimSiam` SSL framework and train it for 250 batches. This will allow the router weights to learn from the data, meaning experts will start to specialize semantically. After training, we can re-run the routing comparison to observe meaningful differences!

In [ ]:
import numpy as np
import torch.optim as optim
from tqdm.auto import tqdm

from cafl4ds.measurements import rankme
from cafl4ds.ssl.factory import build_simsiam

simsiam = build_simsiam(encoder, aux_weight=0.1).to(device)
optimizer = optim.AdamW((p for p in simsiam.parameters() if p.requires_grad), lr=1e-5, weight_decay=1e-4)
history = {"epoch": [], "loss": [], "rankme": []}
allowed_eras = {era for era, name in stream.era_names.items() if any(t in name for t in target_regime_names)}
if not allowed_eras:
    raise ValueError(f"No batches match {target_regime_names}; available: {stream.era_names}")


def training_batches_for_regimes():
    # Repeat eligible batches without materializing another image copy or exhausting a filtered iterator.
    while True:
        found = False
        for batch in stream:
            if batch.era in allowed_eras and len(batch.images) >= 2:
                found = True
                yield batch
        if not found:
            raise ValueError("The requested regimes have no batches of at least two images.")


train_iterator = training_batches_for_regimes()
for epoch in range(training_epochs):
    simsiam.train()
    epoch_losses, all_embeddings = [], []
    for step in tqdm(range(training_batches), desc=f"Epoch {epoch + 1}/{training_epochs}"):
        imgs = next(train_iterator).images.to(device)
        optimizer.zero_grad()
        loss = simsiam.training_step(imgs)
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())
        if step >= max(0, training_batches - 20):
            encoder.eval()
            with torch.no_grad():
                all_embeddings.append(encoder.embed(imgs).cpu())
            simsiam.train()
    if not all_embeddings:
        raise ValueError("training_batches must be positive.")
    rm = rankme(torch.cat(all_embeddings, dim=0))
    history["epoch"].append(epoch + 1)
    history["loss"].append(float(np.mean(epoch_losses)))
    history["rankme"].append(rm)
    logger.info("Epoch {} | Loss: {:.4f} | RankMe: {:.2f}", epoch + 1, history["loss"][-1], rm)
encoder.eval()

In [ ]:
# Plot training performance over epochs
fig, ax1 = plt.subplots(figsize=(8, 5))

color = "tab:red"
ax1.set_xlabel("Epoch")
ax1.set_ylabel("SimSiam Loss", color=color)
ax1.plot(history["epoch"], history["loss"], color=color, marker="o", label="Loss")
ax1.tick_params(axis="y", labelcolor=color)
ax1.set_xticks(history["epoch"])

ax2 = ax1.twinx()
color = "tab:blue"
ax2.set_ylabel("RankMe (Collapse Measure)", color=color)
ax2.plot(history["epoch"], history["rankme"], color=color, marker="s", label="RankMe")
ax2.tick_params(axis="y", labelcolor=color)

fig.tight_layout()
plt.title("Training Performance: Loss and Representation RankMe")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

regimes_to_test = ["daytime", "night", "dawn/dusk"]
expert_counts_per_regime = {}

if stream is not None:
    for regime_name in regimes_to_test:
        try:
            reg_id = next(k for k, v in stream.regime_names.items() if regime_name in v)
            target_era = stream._regime_order.index(reg_id)
            batch = next((b for b in stream if b.era == target_era), None)
            if batch is not None:
                device = next(encoder.parameters()).device
                imgs = batch.images.to(device)
                with torch.no_grad():
                    probs = encoder.routing(imgs)
                # probs is a list of tensors of shape [batch_size * seq_len, num_experts]
                agg_probs = []
                for prob in probs.values():
                    avg_prob = prob.mean(dim=(0, 1)).cpu().numpy()
                    agg_probs.append(avg_prob)
                expert_counts_per_regime[regime_name] = agg_probs
        except (ValueError, StopIteration):
            logger.warning(f"Regime containing '{regime_name}' not found in stream.")

if expert_counts_per_regime:
    num_blocks = len(next(iter(expert_counts_per_regime.values())))
    fig, axes = plt.subplots(1, num_blocks, figsize=(6 * num_blocks, 5))
    if num_blocks == 1:
        axes = [axes]

    for i in range(num_blocks):
        ax = axes[i]
        num_experts = len(next(iter(expert_counts_per_regime.values()))[i])
        x = np.arange(num_experts)
        width = 0.25
        multiplier = 0

        for regime_name, agg_probs in expert_counts_per_regime.items():
            offset = width * multiplier
            rects = ax.bar(x + offset, agg_probs[i], width, label=regime_name)
            multiplier += 1

        ax.set_title(f"Average Expert Probabilities (MoE Block {i})")
        ax.set_xlabel("Expert Index")
        ax.set_ylabel("Average Probability")
        ax.set_xticks(x + width * (len(expert_counts_per_regime) - 1) / 2)
        ax.set_xticklabels([f"E{j}" for j in range(num_experts)])
        ax.legend(loc="upper right")

    plt.tight_layout()
    plt.show()
else:
    print("No valid regimes found for comparison.")

### Routing Probabilities Visualization

We visualize the routing probabilities for each MoE block. A heatmap illustrates how tokens (y-axis) are distributed among the 4 experts (x-axis).

In [ ]:
probs_block2 = probs[encoder.moe_blocks[0]].detach().cpu().numpy()  # [B, T, E]
probs_block3 = probs[encoder.moe_blocks[1]].detach().cpu().numpy()  # [B, T, E]

sample_idx = 0  # Visualize the first image in the batch

# Calculate number of tokens and adjust annotations dynamically
num_tokens = probs_block2.shape[1]
show_annot = num_tokens <= 32
yticklabels = [f"{i}" for i in range(num_tokens)] if show_annot else False
fig_height = max(6, int(num_tokens * 0.3)) if show_annot else 8

fig, axes = plt.subplots(1, 2, figsize=(14, fig_height))

sns.heatmap(
    probs_block2[sample_idx],
    cmap="Blues",
    ax=axes[0],
    cbar_kws={"label": "Probability"},
    annot=show_annot,
    fmt=".2f",
    yticklabels=yticklabels,
)
axes[0].set_title("Encoder Layer 3 Routing (First Image)")
axes[0].set_xlabel("Expert Index")
axes[0].set_ylabel("Token Index (Spatial Patch)")

sns.heatmap(
    probs_block3[sample_idx],
    cmap="Oranges",
    ax=axes[1],
    cbar_kws={"label": "Probability"},
    annot=show_annot,
    fmt=".2f",
    yticklabels=yticklabels,
)
axes[1].set_title("Encoder Layer 4 Routing (First Image)")
axes[1].set_xlabel("Expert Index")
axes[1].set_ylabel("Token Index (Spatial Patch)")

plt.tight_layout()
plt.show()

### Spatial Routing Distribution

We project the expert routing probabilities back to the spatial dimensions of the image. This allows us to see which parts of the image each expert specializes in processing.

In [ ]:
import math

import numpy as np
import torch.nn.functional as F

sample_idx = min(3, len(imgs) - 1)
img = imgs[sample_idx].cpu().detach().cpu().numpy().transpose(1, 2, 0)
# Normalize image for visualization
img = (img - img.min()) / (img.max() - img.min() + 1e-8)

# We visualize the spatial routing for the first MoE block
p_block = probs[encoder.moe_blocks[0]][sample_idx].detach().cpu()  # [T, E]
num_patches = (IMAGE_SIZE // PATCH_SIZE) ** 2
if p_block.shape[0] > num_patches:
    p_block = p_block[-num_patches:]  # Keep only spatial patches if class token is present

E = p_block.shape[1]
grid_size = int(np.sqrt(p_block.shape[0]))
img_size = imgs.shape[2:]

cols = math.ceil(E / 2)
fig, axes = plt.subplots(2, cols, figsize=(3 * cols, 6))
axes = axes.flatten()

# vmax_val = p_block.max().item() # Dynamic vmax for stronger heatmap
vmax_val = 0.5  # Fixed vmax for consistent color scaling across experts

for e in range(E):
    expert_prob = p_block[:, e].view(1, 1, grid_size, grid_size)
    expert_prob_resized = F.interpolate(expert_prob, size=img_size, mode="bilinear", align_corners=False)
    expert_prob_resized = expert_prob_resized.squeeze().numpy()

    ax = axes[e]
    ax.imshow(img)
    im = ax.imshow(expert_prob_resized, cmap="jet", alpha=0.65, vmin=0, vmax=vmax_val)
    ax.set_title(f"Expert {e}")
    ax.axis("off")

# Hide any unused subplots
for e in range(E, len(axes)):
    axes[e].axis("off")

cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cbar_ax, label="Routing Probability")
plt.suptitle("Spatial Routing Distribution for First MoE Block")
plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.show()

## Activation Maximization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch.optim import Adam
from tqdm.auto import tqdm


def maximize_expert_activation(encoder, target_block_idx=0, target_expert_idx=0, img_size=64, iterations=None, lr=0.1):
    """Optimize an input using differentiable probabilities from the actual forward pass."""
    iterations = activation_iterations if iterations is None else iterations
    device = next(encoder.parameters()).device
    modes = {module: module.training for module in encoder.modules()}
    grad_flags = {param: param.requires_grad for param in encoder.parameters()}
    captured = {}

    def capture(module, args, result):
        captured["probabilities"] = result.routing_probs

    block = encoder.blocks[encoder.moe_blocks[target_block_idx]].mlp
    handle = block.register_forward_hook(capture)
    try:
        encoder.eval()
        for param in encoder.parameters():
            param.requires_grad_(False)
        img = (torch.randn(1, 3, img_size, img_size, device=device) * 0.1).requires_grad_(True)
        optimizer = Adam([img], lr=lr, weight_decay=1e-4)
        pbar = tqdm(range(iterations))
        for i in pbar:
            optimizer.zero_grad()
            tv_loss = (img[:, :, :, :-1] - img[:, :, :, 1:]).abs().sum()
            tv_loss = tv_loss + (img[:, :, :-1, :] - img[:, :, 1:, :]).abs().sum()
            encoder(img)
            # routing() is a no-grad diagnostic; the forward hook retains the input gradient.
            target_prob = captured["probabilities"][0, :, target_expert_idx].mean()
            loss = -target_prob + 0.01 * tv_loss
            loss.backward()
            optimizer.step()
            with torch.no_grad():
                img.clamp_(-2.5, 2.5)
            if i % 50 == 0:
                pbar.set_description(f"Expert {target_expert_idx} Prob: {target_prob.item():.4f}")
        result = img.detach().cpu().squeeze(0).permute(1, 2, 0).numpy()
        return (result - result.min()) / (result.max() - result.min() + 1e-8)
    finally:
        handle.remove()
        for param, enabled in grad_flags.items():
            param.requires_grad_(enabled)
        for module, mode in modes.items():
            module.training = mode


# --- Run Activation Maximization for the first MoE Block ---
# Get the actual number of experts from the layer
num_experts = encoder.blocks[encoder.moe_blocks[0]].mlp.num_experts

fig, axes = plt.subplots(1, num_experts, figsize=(16, 4))
fig.suptitle("Activation Maximization: What triggers each expert in the first MoE block?", fontsize=14)

for expert_idx in range(num_experts):
    print(f"\nOptimizing for Expert {expert_idx}...")
    img_opt = maximize_expert_activation(encoder, target_block_idx=0, target_expert_idx=expert_idx, img_size=IMAGE_SIZE)
    axes[expert_idx].imshow(img_opt)
    axes[expert_idx].set_title(f"Expert {expert_idx}")
    axes[expert_idx].axis("off")

plt.tight_layout()
plt.show()

## Mutual Information Test

In [ ]:
import numpy as np
import torch
from sklearn.metrics import mutual_info_score
from tqdm.auto import tqdm


def evaluate_routing_mutual_information(encoder, stream, target_block_idx=0, max_batches=None):
    """Computes the mutual information between the router's expert assignment
    and the image's regime (e.g. time of day) to test for specialization.
    """
    device = next(encoder.parameters()).device
    encoder.eval()

    y_regime = []
    y_expert = []

    print("Evaluating mutual information...")

    batch_iterator = iter(stream)
    for _ in tqdm(range(max_batches) if max_batches is not None else range(len(stream))):
        try:
            batch = next(batch_iterator)
        except StopIteration:
            break

        imgs = batch.images.to(device)
        era = batch.era  # The regime index provided by RegimeStream

        with torch.no_grad():
            # Deterministic, explicit routing readout (no cached module attributes).
            actual_block_idx = encoder.moe_blocks[target_block_idx]
            block_probs = encoder.routing(imgs)[actual_block_idx]

            # Aggregate token-level routing into an image-level expert assignment
            image_probs = block_probs.mean(dim=1)  # Shape: [B, E]
            chosen_experts = image_probs.argmax(dim=-1).cpu().numpy()  # Shape: [B]

        y_regime.extend([era] * imgs.shape[0])
        y_expert.extend(chosen_experts)

    y_regime = np.array(y_regime)
    y_expert = np.array(y_expert)

    # 1. Calculate the Observed Mutual Information
    mi = mutual_info_score(y_regime, y_expert)

    # 2. Calculate the Label-Permutation Null (Random baseline)
    null_mis = []
    for _ in range(100):
        shuffled_regimes = np.random.permutation(y_regime)
        null_mis.append(mutual_info_score(shuffled_regimes, y_expert))

    null_mi_mean = np.mean(null_mis)
    null_mi_std = np.std(null_mis)

    # 3. Check for Routing Collapse
    unique_experts, counts = np.unique(y_expert, return_counts=True)
    max_expert_share = np.max(counts) / len(y_expert)

    print("\n" + "=" * 40)
    print("       MUTUAL INFORMATION RESULTS")
    print("=" * 40)
    print(f"Observed MI:        {mi:.4f} nats")
    print(f"Random Null MI:     {null_mi_mean:.4f} ± {null_mi_std:.4f} nats")
    print("-" * 40)

    # Z-score test
    if null_mi_std > 0:
        z_score = (mi - null_mi_mean) / null_mi_std
        print(f"Z-Score: {z_score:.1f}")

    if max_expert_share > 0.8:
        print("\nWarning: ROUTING COLLAPSE DETECTED!")
        print(f"One expert is processing {max_expert_share * 100:.1f}% of all images.")
        print(
            "The high Z-Score is caused by slight distribution shifts in the remaining rare experts, NOT true specialization."
        )
    elif mi > null_mi_mean + (3 * null_mi_std):
        print("\nConclusion: STRONG EVIDENCE of specialization!")
        print("The router depends heavily on the regime (weather/time of day).")
    else:
        print("\nConclusion: WEAK/NO EVIDENCE of specialization.")
        print("The router assigns experts mostly independently of the regime.")

    return mi, y_regime, y_expert


# Run the test on the first MoE block (moe_blocks[0])
mi_score, regimes, experts = evaluate_routing_mutual_information(encoder, stream, target_block_idx=0, max_batches=None)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


def plot_regime_expert_heatmap(regimes, experts, stream):
    """Visualizes the joint distribution of Regimes and Expert assignments as a heatmap.
    """
    # 1. Map the numerical regime IDs to their actual string names
    regime_names_list = []
    for r in regimes:
        # Fallback to "Regime X" if the name isn't found
        name = stream.era_names.get(r, f"Regime {r}") if hasattr(stream, "regime_names") else f"Regime {r}"
        regime_names_list.append(name)

    # 2. Create a Pandas DataFrame for easy cross-tabulation
    df = pd.DataFrame({"Regime": regime_names_list, "Expert": [f"Expert {e}" for e in experts]})

    # 3. Create a contingency table (counts of how many times each expert was chosen per regime)
    df["Expert"] = pd.Categorical(
        df["Expert"], categories=[f"Expert {i}" for i in range(number_of_experts)], ordered=True
    )
    contingency = pd.crosstab(df["Regime"], df["Expert"], dropna=False)

    # 4. Normalize by row (so each regime sums to 100%) to see the distribution clearly
    contingency_norm = contingency.div(contingency.sum(axis=1), axis=0) * 100

    # 5. Plot the Heatmap
    plt.figure(figsize=(10, 6))
    sns.heatmap(
        contingency_norm, annot=True, fmt=".1f", cmap="Blues", cbar_kws={"label": "% of images routed to expert"}
    )

    plt.title("Expert Specialization: Which Expert handles which Regime?", pad=15, fontsize=14)
    plt.ylabel("Driving Regime", fontsize=12)
    plt.xlabel("Assigned MoE Expert", fontsize=12)

    # Rotate y-labels so they are easy to read
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


# Run the visualization using the arrays outputted from the MI function
plot_regime_expert_heatmap(regimes, experts, stream)

In [ ]:
mi_score, regimes, experts = evaluate_routing_mutual_information(encoder, stream, target_block_idx=1, max_batches=None)

In [ ]:
plot_regime_expert_heatmap(regimes, experts, stream)

## H2-related ablation: batch balancing versus disabled balancing

This checkout implements batch balancing only. The second arm sets both balancing weights to zero; it is **not an EMA experiment**.

In [ ]:
torch.manual_seed(13)  # Paired initialization for the balancing ablation.
number_of_experts = 4

# --- Change this variable to test different Regimes ---
# e.g. 'daytime', 'night', 'dawn/dusk'
target_regime_names = ["daytime", "night", "dawn/dusk"]


# Initialize the MoE Encoder
encoder_unbalanced = MoEViTEncoder(
    img_size=IMAGE_SIZE,
    patch_size=PATCH_SIZE,
    in_chans=3,
    embed_dim=64,
    depth=4,
    num_heads=2,
    mlp_ratio=2.0,
    num_experts=number_of_experts,
    top_k=1,
    moe_blocks=(2, 3),
    routing_level="token",
    balancing_mode="batch",
    importance_weight=0.0,
    load_weight=0.0,
)

initialize_encoder(encoder_unbalanced)
encoder_unbalanced = encoder_unbalanced.to(device)
encoder_unbalanced.eval()


if stream is not None and len(stream) > 0:
    target_era = 0
    if hasattr(stream, "regime_names") and hasattr(stream, "_regime_order"):
        try:
            # stream.regime_names is a dict[int, str]
            reg_id = next(k for k, v in stream.regime_names.items() if target_regime_names[0] in v)
            target_era = stream._regime_order.index(reg_id)
        except (ValueError, StopIteration):
            logger.warning(f"Regime '{target_regime_names}' not found. Using first batch.")

    batch = next((b for b in stream if b.era == target_era), None)
    if batch is None:
        batch = next(iter(stream))
    device = next(encoder_unbalanced.parameters()).device
    imgs = batch.images.to(device)
    logger.info(f"Processing batch for '{target_regime_names}' with {imgs.shape[0]} images.")
else:
    # Fallback to dummy data if dataset is not found
    raise ValueError("Stream failed to load! Please check the BDD100K data path and memory usage.")

# Forward pass to get routing probabilities
with torch.no_grad():
    embeds, aux_loss = encoder_unbalanced.embed_with_aux(imgs)
probs = encoder_unbalanced.routing(imgs)
print(f"Auxiliary balancing loss: {aux_loss:.4f}")

In [ ]:
import numpy as np
import torch.optim as optim
from tqdm.auto import tqdm

from cafl4ds.measurements import rankme
from cafl4ds.ssl.factory import build_simsiam

simsiam = build_simsiam(encoder_unbalanced, aux_weight=0.1).to(device)
optimizer = optim.AdamW((p for p in simsiam.parameters() if p.requires_grad), lr=1e-5, weight_decay=1e-4)
history = {"epoch": [], "loss": [], "rankme": []}
allowed_eras = {era for era, name in stream.era_names.items() if any(t in name for t in target_regime_names)}
if not allowed_eras:
    raise ValueError(f"No batches match {target_regime_names}; available: {stream.era_names}")


def training_batches_for_regimes():
    # Repeat eligible batches without materializing another image copy or exhausting a filtered iterator.
    while True:
        found = False
        for batch in stream:
            if batch.era in allowed_eras and len(batch.images) >= 2:
                found = True
                yield batch
        if not found:
            raise ValueError("The requested regimes have no batches of at least two images.")


train_iterator = training_batches_for_regimes()
for epoch in range(training_epochs):
    simsiam.train()
    epoch_losses, all_embeddings = [], []
    for step in tqdm(range(training_batches), desc=f"Epoch {epoch + 1}/{training_epochs}"):
        imgs = next(train_iterator).images.to(device)
        optimizer.zero_grad()
        loss = simsiam.training_step(imgs)
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())
        if step >= max(0, training_batches - 20):
            encoder_unbalanced.eval()
            with torch.no_grad():
                all_embeddings.append(encoder_unbalanced.embed(imgs).cpu())
            simsiam.train()
    if not all_embeddings:
        raise ValueError("training_batches must be positive.")
    rm = rankme(torch.cat(all_embeddings, dim=0))
    history["epoch"].append(epoch + 1)
    history["loss"].append(float(np.mean(epoch_losses)))
    history["rankme"].append(rm)
    logger.info("Epoch {} | Loss: {:.4f} | RankMe: {:.2f}", epoch + 1, history["loss"][-1], rm)
encoder_unbalanced.eval()

In [ ]:
# Plot training performance over epochs
fig, ax1 = plt.subplots(figsize=(8, 5))

color = "tab:red"
ax1.set_xlabel("Epoch")
ax1.set_ylabel("SimSiam Loss", color=color)
ax1.plot(history["epoch"], history["loss"], color=color, marker="o", label="Loss")
ax1.tick_params(axis="y", labelcolor=color)
ax1.set_xticks(history["epoch"])

ax2 = ax1.twinx()
color = "tab:blue"
ax2.set_ylabel("RankMe (Collapse Measure)", color=color)
ax2.plot(history["epoch"], history["rankme"], color=color, marker="s", label="RankMe")
ax2.tick_params(axis="y", labelcolor=color)

fig.tight_layout()
plt.title("Training Performance: Loss and Representation RankMe")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

regimes_to_test = ["daytime", "night", "dawn/dusk"]
expert_counts_per_regime = {}

if stream is not None:
    for regime_name in regimes_to_test:
        try:
            reg_id = next(k for k, v in stream.regime_names.items() if regime_name in v)
            target_era = stream._regime_order.index(reg_id)
            batch = next((b for b in stream if b.era == target_era), None)
            if batch is not None:
                device = next(encoder_unbalanced.parameters()).device
                imgs = batch.images.to(device)
                with torch.no_grad():
                    probs = encoder_unbalanced.routing(imgs)
                # probs is a list of tensors of shape [batch_size * seq_len, num_experts]
                agg_probs = []
                for prob in probs.values():
                    avg_prob = prob.mean(dim=(0, 1)).cpu().numpy()
                    agg_probs.append(avg_prob)
                expert_counts_per_regime[regime_name] = agg_probs
        except (ValueError, StopIteration):
            logger.warning(f"Regime containing '{regime_name}' not found in stream.")

if expert_counts_per_regime:
    num_blocks = len(next(iter(expert_counts_per_regime.values())))
    fig, axes = plt.subplots(1, num_blocks, figsize=(6 * num_blocks, 5))
    if num_blocks == 1:
        axes = [axes]

    for i in range(num_blocks):
        ax = axes[i]
        num_experts = len(next(iter(expert_counts_per_regime.values()))[i])
        x = np.arange(num_experts)
        width = 0.25
        multiplier = 0

        for regime_name, agg_probs in expert_counts_per_regime.items():
            offset = width * multiplier
            rects = ax.bar(x + offset, agg_probs[i], width, label=regime_name)
            multiplier += 1

        ax.set_title(f"Average Expert Probabilities (MoE Block {i})")
        ax.set_xlabel("Expert Index")
        ax.set_ylabel("Average Probability")
        ax.set_xticks(x + width * (len(expert_counts_per_regime) - 1) / 2)
        ax.set_xticklabels([f"E{j}" for j in range(num_experts)])
        ax.legend(loc="upper right")

    plt.tight_layout()
    plt.show()
else:
    print("No valid regimes found for comparison.")

### Routing Probabilities Visualization

We visualize the routing probabilities for each MoE block. A heatmap illustrates how tokens (y-axis) are distributed among the 4 experts (x-axis).

In [ ]:
probs_block2 = probs[encoder_unbalanced.moe_blocks[0]].detach().cpu().numpy()  # [B, T, E]
probs_block3 = probs[encoder_unbalanced.moe_blocks[1]].detach().cpu().numpy()  # [B, T, E]

sample_idx = 0  # Visualize the first image in the batch

# Calculate number of tokens and adjust annotations dynamically
num_tokens = probs_block2.shape[1]
show_annot = num_tokens <= 32
yticklabels = [f"{i}" for i in range(num_tokens)] if show_annot else False
fig_height = max(6, int(num_tokens * 0.3)) if show_annot else 8

fig, axes = plt.subplots(1, 2, figsize=(14, fig_height))

sns.heatmap(
    probs_block2[sample_idx],
    cmap="Blues",
    ax=axes[0],
    cbar_kws={"label": "Probability"},
    annot=show_annot,
    fmt=".2f",
    yticklabels=yticklabels,
)
axes[0].set_title("Encoder Layer 3 Routing (First Image)")
axes[0].set_xlabel("Expert Index")
axes[0].set_ylabel("Token Index (Spatial Patch)")

sns.heatmap(
    probs_block3[sample_idx],
    cmap="Oranges",
    ax=axes[1],
    cbar_kws={"label": "Probability"},
    annot=show_annot,
    fmt=".2f",
    yticklabels=yticklabels,
)
axes[1].set_title("Encoder Layer 4 Routing (First Image)")
axes[1].set_xlabel("Expert Index")
axes[1].set_ylabel("Token Index (Spatial Patch)")

plt.tight_layout()
plt.show()

### Spatial Routing Distribution

We project the expert routing probabilities back to the spatial dimensions of the image. This allows us to see which parts of the image each expert specializes in processing.

In [ ]:
import math

import numpy as np
import torch.nn.functional as F

sample_idx = min(3, len(imgs) - 1)
img = imgs[sample_idx].cpu().detach().cpu().numpy().transpose(1, 2, 0)
# Normalize image for visualization
img = (img - img.min()) / (img.max() - img.min() + 1e-8)

# We visualize the spatial routing for the first MoE block
p_block = probs[encoder_unbalanced.moe_blocks[0]][sample_idx].detach().cpu()  # [T, E]
num_patches = (IMAGE_SIZE // PATCH_SIZE) ** 2
if p_block.shape[0] > num_patches:
    p_block = p_block[-num_patches:]  # Keep only spatial patches if class token is present

E = p_block.shape[1]
grid_size = int(np.sqrt(p_block.shape[0]))
img_size = imgs.shape[2:]

cols = math.ceil(E / 2)
fig, axes = plt.subplots(2, cols, figsize=(3 * cols, 6))
axes = axes.flatten()

# vmax_val = p_block.max().item() # Dynamic vmax for stronger heatmap
vmax_val = 0.5  # Fixed vmax for consistent color scaling across experts

for e in range(E):
    expert_prob = p_block[:, e].view(1, 1, grid_size, grid_size)
    expert_prob_resized = F.interpolate(expert_prob, size=img_size, mode="bilinear", align_corners=False)
    expert_prob_resized = expert_prob_resized.squeeze().numpy()

    ax = axes[e]
    ax.imshow(img)
    im = ax.imshow(expert_prob_resized, cmap="jet", alpha=0.65, vmin=0, vmax=vmax_val)
    ax.set_title(f"Expert {e}")
    ax.axis("off")

# Hide any unused subplots
for e in range(E, len(axes)):
    axes[e].axis("off")

cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cbar_ax, label="Routing Probability")
plt.suptitle("Spatial Routing Distribution for First MoE Block")
plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.show()

## Activation Maximization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

# --- Run Activation Maximization for the first MoE Block ---
# Get the actual number of experts from the layer
num_experts = encoder_unbalanced.blocks[encoder_unbalanced.moe_blocks[0]].mlp.num_experts

fig, axes = plt.subplots(1, num_experts, figsize=(16, 4))
fig.suptitle("Activation Maximization: What triggers each expert in the first MoE block?", fontsize=14)

for expert_idx in range(num_experts):
    print(f"\nOptimizing for Expert {expert_idx}...")
    img_opt = maximize_expert_activation(
        encoder_unbalanced, target_block_idx=0, target_expert_idx=expert_idx, img_size=IMAGE_SIZE
    )
    axes[expert_idx].imshow(img_opt)
    axes[expert_idx].set_title(f"Expert {expert_idx}")
    axes[expert_idx].axis("off")

plt.tight_layout()
plt.show()

## Mutual Information Test

In [ ]:
import numpy as np
import torch
from tqdm.auto import tqdm

# Run the test on the first MoE block (moe_blocks[0])
mi_score, regimes, experts = evaluate_routing_mutual_information(
    encoder_unbalanced, stream, target_block_idx=0, max_batches=None
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


def plot_regime_expert_heatmap(regimes, experts, stream):
    """Visualizes the joint distribution of Regimes and Expert assignments as a heatmap.
    """
    # 1. Map the numerical regime IDs to their actual string names
    regime_names_list = []
    for r in regimes:
        # Fallback to "Regime X" if the name isn't found
        name = stream.era_names.get(r, f"Regime {r}") if hasattr(stream, "regime_names") else f"Regime {r}"
        regime_names_list.append(name)

    # 2. Create a Pandas DataFrame for easy cross-tabulation
    df = pd.DataFrame({"Regime": regime_names_list, "Expert": [f"Expert {e}" for e in experts]})

    # 3. Create a contingency table (counts of how many times each expert was chosen per regime)
    df["Expert"] = pd.Categorical(
        df["Expert"], categories=[f"Expert {i}" for i in range(number_of_experts)], ordered=True
    )
    contingency = pd.crosstab(df["Regime"], df["Expert"], dropna=False)

    # 4. Normalize by row (so each regime sums to 100%) to see the distribution clearly
    contingency_norm = contingency.div(contingency.sum(axis=1), axis=0) * 100

    # 5. Plot the Heatmap
    plt.figure(figsize=(10, 6))
    sns.heatmap(
        contingency_norm, annot=True, fmt=".1f", cmap="Blues", cbar_kws={"label": "% of images routed to expert"}
    )

    plt.title("Expert Specialization: Which Expert handles which Regime?", pad=15, fontsize=14)
    plt.ylabel("Driving Regime", fontsize=12)
    plt.xlabel("Assigned MoE Expert", fontsize=12)

    # Rotate y-labels so they are easy to read
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


# Run the visualization using the arrays outputted from the MI function
plot_regime_expert_heatmap(regimes, experts, stream)

### Spatial Routing Distribution (Block 3)

In [ ]:
import math

p_block = probs[encoder_unbalanced.moe_blocks[1]][sample_idx].detach().cpu()  # [T, E]
num_patches = (IMAGE_SIZE // PATCH_SIZE) ** 2
if p_block.shape[0] > num_patches:
    p_block = p_block[-num_patches:]  # Keep only spatial patches if class token is present

E = p_block.shape[1]
grid_size = int(np.sqrt(p_block.shape[0]))
img_size = imgs.shape[2:]

cols = math.ceil(E / 2)
fig, axes = plt.subplots(2, cols, figsize=(3 * cols, 6))
axes = axes.flatten()

vmax_val = 0.5

for e in range(E):
    expert_prob = p_block[:, e].view(1, 1, grid_size, grid_size)
    expert_prob_resized = F.interpolate(expert_prob, size=img_size, mode="bilinear", align_corners=False)
    expert_prob_resized = expert_prob_resized.squeeze().numpy()

    ax = axes[e]
    ax.imshow(img)
    im = ax.imshow(expert_prob_resized, cmap="jet", alpha=0.65, vmin=0, vmax=vmax_val)
    ax.set_title(f"Expert {e}")
    ax.axis("off")

for e in range(E, len(axes)):
    axes[e].axis("off")

cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cbar_ax, label="Routing Probability")
plt.suptitle("Spatial Routing Distribution for Second MoE Block")
plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.show()

### Activation Maximization (Block 3)

In [ ]:
# --- Run Activation Maximization for the second MoE Block ---
num_experts = encoder_unbalanced.blocks[encoder_unbalanced.moe_blocks[1]].mlp.num_experts

fig, axes = plt.subplots(1, num_experts, figsize=(16, 4))
fig.suptitle("Activation Maximization: What triggers each expert in the second MoE block?", fontsize=14)

for expert_idx in range(num_experts):
    print(f"\nOptimizing for Expert {expert_idx}...")
    img_opt = maximize_expert_activation(
        encoder_unbalanced, target_block_idx=1, target_expert_idx=expert_idx, img_size=IMAGE_SIZE
    )
    axes[expert_idx].imshow(img_opt)
    axes[expert_idx].set_title(f"Expert {expert_idx}")
    axes[expert_idx].axis("off")

plt.tight_layout()
plt.show()

### Mutual Information Test (Block 3)

In [ ]:
# Run the test on the second MoE block (moe_blocks[1])
mi_score_b3, regimes_b3, experts_b3 = evaluate_routing_mutual_information(
    encoder_unbalanced, stream, target_block_idx=1, max_batches=None
)

In [ ]:
# 1. Get the regime names
regime_names_list = []
for r in regimes_b3:
    try:
        regime_names_list.append(stream.era_names[r])
    except (KeyError, AttributeError):
        regime_names_list.append(str(r))

# 2. Create a Pandas DataFrame for easy cross-tabulation
df = pd.DataFrame({"Regime": regime_names_list, "Expert": [f"Expert {e}" for e in experts_b3]})

df["Expert"] = pd.Categorical(df["Expert"], categories=[f"Expert {i}" for i in range(num_experts)], ordered=True)
contingency = pd.crosstab(df["Regime"], df["Expert"], dropna=False)
contingency_pct = contingency.div(contingency.sum(axis=1), axis=0) * 100

plt.figure(figsize=(10, 6))
sns.heatmap(contingency_pct, annot=True, fmt=".1f", cmap="Blues", cbar_kws={"label": "% of images routed to expert"})
plt.title("Expert Specialization for Second MoE Block")
plt.ylabel("Driving Regime")
plt.xlabel("Assigned MoE Expert")
plt.show()

## Interpreting the experiments

Routing plots and mutual information are exploratory measurements. Re-run all cells with the intended dataset and checkpoint before drawing conclusions. A random-start run does not support pretrained-backbone claims. The second arm disables batch balancing; EMA balancing is not implemented in this repository, so the original EMA version of H2 remains untested.
